# 1.1 — Build Interim Application Table

**Case**: Credit granting policy review for a Fintech client (Klavi).

**Goal of this notebook**: turn the two raw files into **one clean table, one row per application**
(`uuid` + `DATA`), carrying the feature snapshot taken at the application date
(`REFERENCIA` = `DATA` — snapshots are application-triggered, so a simple left join suffices;
verified in §3), with sentinels cleaned and explicit coverage/segment flags —
written to `data/interim/applications.parquet`.

**Inputs** (`data/raw/`): `profile_case_credit.parquet` (performance), `transactions_case_credit.parquet`
(application features) — see notebook 1.0 for the full exploration.

**Cleaning decisions** (carried over from 1.0 §8):
- `score_klavi_1`: -999999 sentinel → NaN (real range 0–100)
- `status_empregaticio`: `'-999999'` label → NaN (kept distinct from the provider-given `Desconhecido`)
- `Capacidade Financeira`: **negatives kept as valid** (over-committed is a real state);
  only the extreme tail gets a boolean quality flag — winsorizing deferred to phase 2
- `Renda`: untouched here (thin extreme tail handled in phase-2 feature analysis)

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

PROJECT_DIR = Path.cwd().parent
RAW_DIR = PROJECT_DIR / "data" / "raw"
INTERIM_DIR = PROJECT_DIR / "data" / "interim"

PROFILE_PATH = RAW_DIR / "profile_case_credit.parquet" # performance data
TRANSACTIONS_PATH = RAW_DIR / "transactions_case_credit.parquet" # application features
OUT_PATH = INTERIM_DIR / "applications.parquet"


## 1. Load & dtype normalization

Same casts as notebook 1.0 (in-memory; raw files stay untouched).

In [2]:
perf = pd.read_parquet(PROFILE_PATH)      # performance / target table
feat = pd.read_parquet(TRANSACTIONS_PATH)  # features table

# drop leftover pandas index column if present
feat = feat.drop(columns=[c for c in feat.columns if c.startswith("__index_level")])

# in-memory conversions (same as 1.0)
perf["DATA"] = pd.to_datetime(perf["DATA"])
perf["SAFRA"] = perf["SAFRA"].astype(int)
feat["REFERENCIA"] = pd.to_datetime(feat["REFERENCIA"])
feat["risco_aposta"] = feat["risco_aposta"].astype("Int64").astype("string")

print(f"perf: {perf.shape[0]:,} rows x {perf.shape[1]} cols")
print(f"feat: {feat.shape[0]:,} rows x {feat.shape[1]} cols")
print("perf:", dict(perf.dtypes.astype(str)))
print("feat:", dict(feat.dtypes.astype(str)))

perf: 220,618 rows x 5 cols
feat: 371,596 rows x 10 cols
perf: {'uuid': 'str', 'DATA': 'datetime64[us]', 'SAFRA': 'int64', 'OVER30M2': 'float64', 'STATUS': 'str'}
feat: {'uuid': 'str', 'REFERENCIA': 'datetime64[us]', 'score_klavi_1': 'float64', 'score_klavi_2': 'float64', 'classe_social': 'str', 'status_empregaticio': 'str', 'regiao': 'str', 'Renda': 'float64', 'risco_aposta': 'string', 'Capacidade Financeira': 'float64'}


**Key takeaway**: 220,618 perf rows and 371,596 feat rows loaded with the 1.0 casts applied
(`DATA`/`REFERENCIA` → datetime, `SAFRA` → int, `risco_aposta` → categorical string); raw files untouched.

## 2. Clean sentinels & invalid values

Applied on `feat` **before** merging, so the interim table never carries a sentinel.
The before/after missingness table at the end shows exactly what moved to NaN — and that nothing else changed.

In [3]:
missing_before = feat.isna().sum()

# score_klavi_1: -999999 sentinel -> NaN; real range is 0-100
n_sk1 = (feat["score_klavi_1"] == -999999).sum()
feat.loc[feat["score_klavi_1"] == -999999, "score_klavi_1"] = np.nan
assert feat["score_klavi_1"].dropna().between(0, 100).all(), "score_klavi_1 outside [0, 100] after cleaning"

# status_empregaticio: literal '-999999' label -> NaN (distinct from provider-given 'Desconhecido')
n_emp = (feat["status_empregaticio"] == "-999999").sum()
feat.loc[feat["status_empregaticio"] == "-999999", "status_empregaticio"] = pd.NA

print(f"score_klavi_1 sentinels -> NaN: {n_sk1:,}")
print(f"status_empregaticio '-999999' -> NaN: {n_emp:,}")

score_klavi_1 sentinels -> NaN: 2,341
status_empregaticio '-999999' -> NaN: 3,417


In [4]:
# Capacidade Financeira: negatives are kept as valid (over-committed applicants are a real state).
# Only the extreme tail (below p0.1 of non-null values) gets a boolean quality flag.
cf = feat["Capacidade Financeira"]
CF_EXTREME_CUTOFF = cf.quantile(0.001)
feat["flag_capacidade_extrema"] = cf < CF_EXTREME_CUTOFF

print(f"Capacidade Financeira negatives kept: {(cf < 0).sum():,}")
print(f"extreme cutoff (p0.1): {CF_EXTREME_CUTOFF:,.0f}  ->  flagged rows: {feat['flag_capacidade_extrema'].sum():,}")

# Renda: untouched here — thin extreme tail (323 rows above p99) is a phase-2 decision
p99 = feat["Renda"].quantile(0.99)
print(f"Renda untouched; rows above p99 ({p99:,.0f}): {(feat['Renda'] > p99).sum():,}")

Capacidade Financeira negatives kept: 22,005
extreme cutoff (p0.1): -12,414  ->  flagged rows: 317
Renda untouched; rows above p99 (16,533): 323


In [5]:
# before/after: only score_klavi_1 and status_empregaticio may move
missing_after = feat.drop(columns="flag_capacidade_extrema").isna().sum()
delta = (missing_after - missing_before).rename("new_NaN")
pd.concat([missing_before.rename("missing_before"), missing_after.rename("missing_after"), delta], axis=1)

,missing_before,missing_after,new_NaN
uuid,0,0,0
REFERENCIA,0,0,0
score_klavi_1,339319,341660,2341
score_klavi_2,40023,40023,0
classe_social,0,0,0
status_empregaticio,0,3417,3417
regiao,583,583,0
Renda,339319,339319,0
risco_aposta,339319,339319,0
Capacidade Financeira,46310,46310,0


**Key takeaway**: 2,341 `score_klavi_1` sentinels and 3,417 `status_empregaticio` `'-999999'` labels
went to NaN — no other column changed. `score_klavi_1` now lies in [0, 100]. Negative
`Capacidade Financeira` kept as valid signal, with `flag_capacidade_extrema` marking the tail below p0.1.

## 3. Merge on application date & key uniqueness

Snapshots turn out to be **application-triggered**: whenever a `uuid` has any snapshot, it has one
dated exactly at the application date — so a simple left join on `uuid` + `DATA` = `REFERENCIA`
replaces the point-in-time asof rule used during exploration (verified equivalent: both match the
same 114,689 rows and pick the same snapshot). The asserts below are the contract of this notebook:
one row per application, key unique, snapshot date equals application date.

In [6]:
df = perf.merge(
    feat,
    left_on=["uuid", "DATA"], right_on=["uuid", "REFERENCIA"],
    how="left",
)

# contract: one row per application, unique key, snapshot taken at the application date
assert len(df) == len(perf), f"row count changed: {len(df):,} vs {len(perf):,}"
assert not df.duplicated(subset=["uuid", "DATA"]).any(), "uuid + DATA not unique"
matched = df["REFERENCIA"].notna()
assert (df.loc[matched, "REFERENCIA"] == df.loc[matched, "DATA"]).all(), "snapshot date != application date"

# equivalence with the asof rule: the exact-date join reaches every application whose uuid
# has any snapshot at all — no application depends on an older snapshot
assert matched.sum() == perf["uuid"].isin(feat["uuid"]).sum(), "an application relies on an older snapshot"

print(f"applications: {len(df):,}  |  with snapshot: {matched.sum():,} ({matched.mean():.1%})")

applications: 220,618  |  with snapshot: 114,689 (52.0%)


**Key takeaway**: the merge preserves exactly one row per application — 220,618 rows, `uuid` + `DATA`
unique, and every matched snapshot is dated exactly at the application date (`REFERENCIA` = `DATA`).
The exact-date left join reaches every application whose `uuid` has any snapshot, so it is equivalent
to — and simpler than — the asof rule. 114,689 applications (52.0%) carry a snapshot.

## 4. Coverage & segment flags

Explicit columns for the segments phase 2 will treat separately: no Klavi data at all,
Klavi data without open-finance consent, and full open-finance data. (Snapshot staleness is a
non-issue: `REFERENCIA` = `DATA` whenever present, so features are contemporaneous by construction.)

In [7]:
df["has_snapshot"] = df["REFERENCIA"].notna()

# the open-finance trio (score_klavi_1 / Renda / risco_aposta) must move together post-clean...
of_cols = ["score_klavi_1", "Renda", "risco_aposta"]
of_notna = df[of_cols].notna()
print("open-finance trio present-together consistency:")
print(of_notna.sum())
# ...except for the sentinel rows of score_klavi_1 that became NaN in section 2:
# Renda defines the segment (untouched by cleaning)
df["has_open_finance"] = df["Renda"].notna()

segments = pd.Series(
    np.select(
        [~df["has_snapshot"], ~df["has_open_finance"]],
        ["no snapshot", "snapshot, no open finance"],
        default="snapshot + open finance",
    )
).value_counts()
print()
print(segments.to_frame("n").assign(pct=lambda t: (t["n"] / len(df) * 100).round(1)))

open-finance trio present-together consistency:
score_klavi_1    9158
Renda            9834
risco_aposta     9834
dtype: int64

                                n   pct
no snapshot                105929  48.0
snapshot, no open finance  104855  47.5
snapshot + open finance      9834   4.5


**Key takeaway**: segment flags in place — `has_snapshot` and `has_open_finance` (defined on `Renda`,
which the cleaning never touched). These make the phase-2 "treat missingness as segments, not just
NaN" strategy a simple groupby. No staleness column is needed: snapshots are always dated at the
application date.

## 5. Final schema & write

One row per application, all STATUS kept (phase 3's swap-set needs the full through-the-door
population). Written to `data/interim/applications.parquet` and re-read to verify the round trip.

In [8]:
schema = pd.DataFrame({"dtype": df.dtypes.astype(str), "n_missing": df.isna().sum(),
                       "pct_missing": (df.isna().mean() * 100).round(2)})
schema

,dtype,n_missing,pct_missing
uuid,str,0,0.00
DATA,datetime64[us],0,0.00
SAFRA,int64,0,0.00
OVER30M2,float64,196470,89.05
STATUS,str,0,0.00
REFERENCIA,datetime64[us],105929,48.01
score_klavi_1,float64,211460,95.85
score_klavi_2,float64,118357,53.65
classe_social,str,105929,48.01
status_empregaticio,str,107176,48.58


In [9]:
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUT_PATH, index=False)

check = pd.read_parquet(OUT_PATH)
assert check.shape == df.shape, "round-trip shape mismatch"
assert not check.duplicated(subset=["uuid", "DATA"]).any(), "round-trip key not unique"
assert dict(check.dtypes.astype(str)) == dict(df.dtypes.astype(str)), "round-trip dtype mismatch"

print(f"written: {OUT_PATH.relative_to(PROJECT_DIR)}  shape={check.shape}  key=uuid+DATA (unique)")

written: data/interim/applications.parquet  shape=(220618, 17)  key=uuid+DATA (unique)


**Key takeaway**: `data/interim/applications.parquet` written — 220,618 rows, one per application,
key `uuid` + `DATA`, dtypes and shape verified on re-read.

## 6. Key takeaways & general findings

**Key takeaways per section**

1. **§1 Load** — 220,618 perf rows and 371,596 feat rows loaded with the 1.0 dtype casts re-applied; raw files untouched.
2. **§2 Cleaning** — 2,341 `score_klavi_1` sentinels and 3,417 `status_empregaticio` `'-999999'` labels → NaN, nothing else altered (verified by before/after missingness deltas); `score_klavi_1` asserted in [0, 100]; negative `Capacidade Financeira` kept as valid with `flag_capacidade_extrema` marking the tail below p0.1.
3. **§3 Merge** — snapshots are application-triggered (`REFERENCIA` = `DATA` whenever present), so a simple left join on `uuid` + application date yields exactly one row per application; key uniqueness, date equality and equivalence with the exploratory asof rule are all asserted; 114,689 applications (52.0%) carry a snapshot.
4. **§4 Segments** — `has_snapshot` and `has_open_finance` (defined on `Renda`, untouched by cleaning) make the phase-2 missingness segments explicit columns; no staleness measure needed since features are contemporaneous by construction.
5. **§5 Output** — `data/interim/applications.parquet`, 220,618 rows, all STATUS kept, round-trip verified.

**General findings**

- The cleaning is **conservative by design**: only provable sentinels went to NaN; distribution-shaping decisions (winsorizing `Renda` / `Capacidade Financeira` tails) are deferred to phase-2 feature analysis where their impact can be measured against the target.
- After sentinel cleaning, `score_klavi_1` availability no longer exactly matches `Renda`/`risco_aposta` (the 2,341 sentinel rows lost their value) — which is why `has_open_finance` is anchored on `Renda`.